In [22]:
# Getting Gemini API and Food API secrets
import os
from google.cloud import secretmanager

project_id = os.environ.get('GOOGLE_CLOUD_PROJECT') # Or your specific project ID

def access_secret_version(secret_id, version_id="latest"): # Function to access secrets
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    return response.payload.data.decode("UTF-8").strip()

try:
    # Access Capstone_project_key
    Capstone_project_key = access_secret_version("Capstone_project_key3") #Capstone_project_key3 : Vetex AI Studio API key
    os.environ["GOOGLE_API_KEY"] = Capstone_project_key
    os.environ["Capstone_project_key"] = Capstone_project_key
    print("✅ Gemini API key setup complete from Secret Manager.")
except Exception as e:
    print(f"🔑 Authentication Error for Capstone_project_key: {e}")

try:
    # Access Food_API
    Food_API = access_secret_version("Food_API")
    os.environ["Food_API"] = Food_API
    print("✅ Food API key setup complete from Secret Manager.")
except Exception as e:
    print(f"🔑 Authentication Error for Food_API: {e}")

✅ Gemini API key setup complete from Secret Manager.
✅ Food API key setup complete from Secret Manager.


In [2]:
#importing ADK components
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types


print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [3]:
#retry config to feed into Agent parameters
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)
print("✅ Retry config for Agents created successfully.")

✅ Retry config for Agents created successfully.


In [4]:
#AlertAgent
from google.adk.tools import AgentTool
# AlertAgent = Agent(
#     name='AlertAgent',
#     model=Gemini(model='gemini-2.5-flash-lite', retry_options=retry_config),
#     instruction='You are an agent that sends alerts.',
#     tools=[]
# )
from datetime import datetime

def get_time() -> str:
    """Get the current time."""
    return datetime.now().isoformat()

AlertAgent = Agent(
    name='AlertAgent',
    model=Gemini(model='gemini-2.5-flash-lite', retry_options=retry_config),
    description="You are a proactive Alert agent",
    instruction="""
    You are a proactive Alert Agent. Your task is to generate alerts for patients who are on medication to take their medication. 
    Use the current time and generate alerts based on the following rules:
    For pre-meal medications, generate alerts if patient's usual breakfast, lunch, and dinner time is in the next 15 minutes
    For once a day medications, generate an alert at their preferred time of the day. 
    For users who take long-acting insulin every night, generate an alert every night at the patients preffered time. 
    For users who take weekly GLP-1 Agonists, generate an alert every week at the patients preferred time of the week."""
   
)
print("✅ AlertAgent created.")

✅ AlertAgent created.


In [40]:
#InsulinAgent
from google.adk.tools import AgentTool
# InsulinAgent = Agent(
#     name='InsulinAgent',
#     model=Gemini(model='gemini-2.5-flash-lite', retry_options=retry_config),
#     instruction='You are an agent that provides insulin recommendations.',
#     tools=[]
# )

#InsulinAgent
def get_insulin_dose(glucose_level: int) -> dict:  
    """Returns the recommended insulin dose based on the glucose level."""
    if glucose_level < 151:
        return {"status": "success", "dose": "No insulin needed"}
    elif 151 <= glucose_level <= 200:
        return {"status": "success", "dose": "Take 2 units of short acting insulin before meal"}
    elif 201 <= glucose_level <= 250:
        return {"status": "success", "dose": "Take 4 units of short acting insulin before meal"}
    elif 251 <= glucose_level <= 300:
        return {"status": "success", "dose": "Take 6 units of short acting insulin before meal"}
    elif 301 <= glucose_level <= 350:
        return {"status": "success", "dose": "Take 8 units of short acting insulin before meal"}
    elif 351 <= glucose_level <= 400:
        return {"status": "success", "dose": "Take 10 units of short acting insulin before meal"}
    else:
        return {"status": "error", "message": "Glucose level too high, please call doctor"}

InsulinAgent = Agent(
    name='InsulinRecommenderAgent',
    model=Gemini(model='gemini-2.5-flash-lite', retry_options=retry_config),
    description="You are an expert Insulin coach. Given a patients glucose level at preferred meal time or current glucose level if the preferred time has passed and there is no indication that the expected meal is taken, provide a suggestion of recommended dosage of insulin.",
    instruction="""
    You are an Insulin Recommender Agent. Your task is to provide a suggestion 
    of an appropriate insulin dosage based on the patient's glucose level.
    
    WORKFLOW:
    1. Call the get_insulin_dose tool with the glucose_level provided
    2. After receiving the tool result, YOU MUST respond with a text message
    
    CRITICAL: You MUST always produce a text response after calling get_insulin_dose.
    Never return an empty response. Never return None.
    
    Your response MUST follow this exact format:
    "Recommended dose: {dose from tool}. Administer before meal."
    
    Example:
    Tool returns: "Take 2 units of short acting insulin before meal"
    Your response: "Recommended dose: Take 2 units of short acting insulin before meal. 
    Administer before meal."
    
    If the tool returns no dose or an error:
    Your response: "Recommended dose: 0 units. No insulin required at this glucose level."
    
    NEVER skip the text response step. NEVER return empty content.
    """,
    tools=[get_insulin_dose],
)
print("✅ InsulinAgent created.")

✅ InsulinAgent created.


In [6]:
# Tool: Food API
API_KEY = Food_API
import requests
#@trace_execution`"search_food_by_carbs")
def search_food_by_carbs(food_name: str, max_carbs: float):
    
    #MealAgent_logger.info(f"Tool called: search_food_by_carbs | food={food_name} | max_carbs={max_carbs}")

    url = "https://api.nal.usda.gov/fdc/v1/foods/search"

    params = {
        "query": food_name,
        "pageSize": 20,
        "api_key": API_KEY
    }

    r = requests.get(url, params=params).json()

    foods = []

    for food in r["foods"]:
        nutrients = food.get("foodNutrients", [])
        nutrient_map = {n["nutrientName"]: n["value"] for n in nutrients}

        carbs = nutrient_map.get("Carbohydrate, by difference")
        protein = nutrient_map.get("Protein")
        calories = nutrient_map.get("Energy (kcal)") or nutrient_map.get("Energy")

        calories_from_carbs = carbs * 4 if carbs is not None else None

        if carbs is not None and carbs <= max_carbs:
            foods.append({
                "name": food["description"],
                "carbs_g": carbs,
                "protein_g": protein,
                "calories_kcal": calories,
                "calories_from_carbs": calories_from_carbs,
                "serving_size": food.get("servingSize"),
                "serving_unit": food.get("servingSizeUnit")
            })
    #MealAgent_logger.info(f"Tool result count: {len(foods)} foods returned")


    return foods

print("✅ search_food_by_carbs()created successfully")

✅ search_food_by_carbs()created successfully


In [43]:
#MealAgent

MealAgent =  Agent(
name= "MealRecommenderAgent",
model=Gemini(
    model="gemini-2.5-flash-lite",
    api_key=Capstone_project_key,   
    retry_options=retry_config
),
description= "Recommends a meal for diabetes management. Recommended meal includes Protein, Vegetables, and Carbohydrates.",
# This instruction tells the Meal Agent HOW to use its tools (which are the other agents).
instruction="""

Role

You are a Diabetes Nutrition Coach Agent.
Your goal is to recommend meals and hydration strategies that help keep the user's
blood glucose within 90–150 mg/dL.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CRITICAL RULE — TOOL USE IS MANDATORY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

You MUST ALWAYS call search_food_by_carbs before recommending any food.
You are NEVER allowed to recommend specific foods from your own knowledge.
Every food item in your final response MUST come from a search_food_by_carbs result.
Responding with food names without calling the tool first is an ERROR.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STEP 1 — DETERMINE GLUCOSE STATUS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Read current_glucose from the input:

  current_glucose > 150  → HIGH:   protein + vegetables ONLY. No carbohydrates.
  current_glucose 70–150 → NORMAL: balanced meal with controlled carbs allowed.
  current_glucose < 70   → LOW:    fast-acting carbohydrates REQUIRED immediately.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STEP 2 — READ AND APPLY DIET PREFERENCE (MANDATORY)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Read the diet preference from the input. Apply these rules strictly:

  Non-Veg (omnivore):
    → Protein options: chicken breast, turkey, fish (salmon, tuna), eggs, Greek yogurt
    → All vegetables and controlled carbs allowed

  Veg (vegetarian, no meat/seafood):
    → Protein options: eggs, Greek yogurt, paneer, tofu, lentils, chickpeas, cottage cheese
    → No chicken, turkey, fish, or any meat/seafood
    → All vegetables and controlled carbs allowed

  Vegan (no animal products):
    → Protein options: tofu, tempeh, lentils, chickpeas, black beans, edamame
    → No meat, seafood, eggs, dairy, or any animal-derived products
    → All vegetables and controlled carbs allowed

  Gluten-Free:
    → Avoid wheat, barley, rye, oats (unless certified gluten-free)
    → Safe carbs: rice, quinoa, sweet potato, corn
    → Can be combined with Non-Veg / Veg / Vegan preference

CRITICAL:
  - If diet = "Veg" or "Vegan" → you MUST NOT search for or recommend
    chicken, turkey, fish, beef, or any meat/seafood under any circumstance
  - If diet = "Vegan" → you MUST NOT search for or recommend eggs, 
    Greek yogurt, paneer, or any dairy product
  - Diet preference OVERRIDES all other food suggestions
  - If search returns a non-compliant food → discard it and search for 
    a compliant alternative

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STEP 3 — IDENTIFY MEAL TYPE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Based on current_time and usual_meal_times, identify:
Breakfast, Lunch, or Dinner.
Always state the meal type explicitly in your response.

Do NOT recommend a meal if the user ate within the last 1 hour,
unless glucose < 70 mg/dL (hypoglycemia always overrides).
You may still recommend hydration even if no meal is needed.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STEP 4 — CALL search_food_by_carbs (MANDATORY)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Use the diet-compliant food options from Step 2 to form your search queries.

  IF glucose > 150 (HIGH — protein + veg only):

    Non-Veg:
      → search_food_by_carbs(food_name="chicken breast", max_carbs=5)
      → search_food_by_carbs(food_name="broccoli", max_carbs=10)

    Veg:
      → search_food_by_carbs(food_name="paneer", max_carbs=5)
      → search_food_by_carbs(food_name="spinach", max_carbs=10)

    Vegan:
      → search_food_by_carbs(food_name="tofu", max_carbs=5)
      → search_food_by_carbs(food_name="broccoli", max_carbs=10)

  IF glucose 70–150 (NORMAL — balanced meal):

    Non-Veg:
      → search_food_by_carbs(food_name="chicken breast", max_carbs=5)
      → search_food_by_carbs(food_name="brown rice", max_carbs=30)
      → search_food_by_carbs(food_name="spinach", max_carbs=10)

    Veg:
      → search_food_by_carbs(food_name="lentils", max_carbs=20)
      → search_food_by_carbs(food_name="brown rice", max_carbs=30)
      → search_food_by_carbs(food_name="spinach", max_carbs=10)

    Vegan:
      → search_food_by_carbs(food_name="tofu", max_carbs=5)
      → search_food_by_carbs(food_name="quinoa", max_carbs=30)
      → search_food_by_carbs(food_name="broccoli", max_carbs=10)

  IF glucose < 70 (LOW — fast-acting carbs, any diet):
      → search_food_by_carbs(food_name="orange juice", max_carbs=30)
      → search_food_by_carbs(food_name="banana", max_carbs=30)

Rules:
  - If search returns {} → try a different diet-compliant food name
  - Never use food data from your training knowledge
  - Never recommend a food that violates the diet preference

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STEP 5 — APPLY DECISION RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Hypoglycemia (glucose < 70 mg/dL):
  - Recommend 15g fast-acting carbohydrates from tool results
  - Wait 15 minutes → recheck glucose
  - If still below 70 → repeat 15g treatment
  - After glucose > 70 → recommend balanced diet-compliant meal

Hyperglycemia hydration (glucose > 180 mg/dL):
  - Recommend 500mL–1L of water

Carbohydrate impact rule:
  - Every 10g carbohydrates raises glucose ~30–50 mg/dL
  - Select portions keeping predicted glucose within 90–150 mg/dL

Medication timing:
  - If user recently took insulin or oral medication →
    recommend eating 15 minutes after medication intake

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
STEP 6 — BUILD RESPONSE FROM TOOL RESULTS ONLY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Use ONLY foods returned by search_food_by_carbs.
Include per food: name, serving size, carbs_g, protein_g, calories_kcal.
Do not invent or estimate nutritional values.
Always confirm diet compliance before including a food in the response.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT FORMAT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Diet Preference: [Non-Veg / Veg / Vegan / Gluten-Free]

Glucose Status: [High / Normal / Low]

Hydration Recommendation:
[specific amount and reason, or "None required"]

Meal Recommendation ([Breakfast / Lunch / Dinner]):

Protein:
[food from tool] — [serving size]g | Protein: [protein_g]g | 
Carbs: [carbs_g]g | Calories: [calories_kcal] kcal

Vegetables:
[food from tool] — [serving size]g | Carbs: [carbs_g]g | 
Calories: [calories_kcal] kcal

Carbohydrates:
[food from tool and portion] — OR —
"None: glucose above target range, carbohydrates withheld"

Estimated Total Carbohydrates: [sum] grams

Estimated glucose impact:
This meal provides an expected [X] mg/dL change, bringing blood glucose to 
approximately [current_glucose + impact] mg/dL within the target of 90–150 mg/dL.

Additional Guidance:
[recheck glucose / medication timing / any relevant note]
""",
    tools=[search_food_by_carbs]
)

#MealAgent_logger.info(f"MealAgent Output: {MealAgent_recommendation}")
#MealAgent = create_meal_agent()
print("✅ MealAgent created.")

✅ MealAgent created.


In [8]:
#ExerciseAgent
from google.adk.tools import AgentTool
import pandas as pd

# ExerciseAgent = Agent(
#     name='ExerciseAgent',
#     model=Gemini(model='gemini-2.5-flash-lite', retry_options=retry_config),
#     instruction='You are an agent that provides exercise recommendations.',
#     tools=[]
# )

def get_exercise_intensity(glucose_level:int) -> list:
    """Returns the recommended exercise intensity based on the glucose level."""
    if glucose_level < 90:
        return ["Avoid"]
    elif 90 <= glucose_level <= 124:
        return ["Light"]
    elif 125 <= glucose_level <= 180:
        return ["Light", "Moderate", "Vigorous"]
    elif 181 <= glucose_level <= 270:
        return ["Light", "Moderate"]
    elif glucose_level > 270:
        return ["Avoid"]

def search_exercise_by_intensity(intensity: str) -> list:
    df = pd.read_csv("traincalc-met-values-latest.csv")
    filtered = df[df["Intensity"] == intensity]
    return filtered[["Description", "MET"]].to_dict(orient="records")

def get_exercise_recommendation(glucose_level: int, meal_carbs: int = None) -> dict:
    # calculating exercise based on current glucose, next steps incorporate pre/post meal exercise recommendations based on carbs in the meal
    intensity_levels = get_exercise_intensity(glucose_level)

    if "Avoid" in intensity_levels:
        return {
            "status": "unsafe",
            "message": "Glucose level not suitable for exercise",
            "pre_exercise": "Consume carbohydrates and recheck glucose"
        }

    all_exercises = []

    for level in intensity_levels:
        exercises = search_exercise_by_intensity(level)

        for e in exercises:
            all_exercises.append({
                "description": e["Description"],
                "met": e["MET"],
                "intensity": level
            })

    return {
        "status": "ok",
        "recommended_exercises": all_exercises
    }

ExerciseAgent = Agent(
    name='ExerciseRecommenderAgent',
    model=Gemini(model='gemini-2.5-flash-lite', retry_options=retry_config),
    description="Given glucose level, recommend appropriate exercise",
    instruction="""
    You are an expert Diabetes Exercise Coach. Your task is to recommend appropriate exercises based on the patient's glucose level. Use the provided get_exercise_recommendation tool 
    to determine suitable exercises based on the glucose level. Then give a recommendation of exercise types, specific exercises, and duration that the patient 
    can do to help manage their blood glucose levels effectively. Be sure to consider the patient's safety and recommend NO exercise if glucose levels are too low or too high. 
    Use the following guidelines for your recommendations:
    Lower than 90 mg/dL (5.0 mmol/L). Your blood sugar may be too low to exercise safely. Before you work out, have a small snack that includes 15 to 30 grams of carbohydrates. 
    Some examples are fruit juice, fruit and crackers. Or take 10 to 20 grams of glucose products, which come in forms such as gels, powders and tablets. 
    After you exercise, check your blood sugar again to find out if it's about 90 mg/dL.
    90-124 mg/dL (5-6.9 mmol/L). Take 10 grams of glucose before you exercise.
    126-180 mg/dL (7-10 mmol/L). You're ready to exercise. But be aware that blood sugar may rise if you do strength training. 
    Blood sugar also may rise if you do short bursts of hard aerobic exercise known as high-intensity interval training.
    182-270 mg/dL (10.2-15 mmol/L). It's okay to exercise. Be aware that blood sugar may rise if you do strength training or high-intensity interval training.
    Over 270 mg/dL (15 mmol/L). This is a caution zone. Your blood sugar may be too high to exercise safely. 
    Before you work out, test your urine for substances called ketones. The body makes ketones when it breaks down fat for energy. 
    The presence of ketones suggests that your body doesn't have enough insulin to control your blood sugar.""",
    tools=[get_exercise_recommendation],
)
print("✅ ExerciseAgent created.")

✅ ExerciseAgent created.


In [19]:
#SafetyGuardAgent

SafetyAgent = Agent(
    name="SafetyGuard",
    model=Gemini(model='gemini-2.5-flash', retry_options=retry_config),
    output_key="judge_output",
    description ="Evaluates the output of Main_agent",
    instruction="""
You are a clinical safety validation agent for a glucose management system.

You will receive a JSON object where Output_Summary is a structured object (not plain text).

Directly read these fields from Output_Summary:
  - user_information
  - current_glucose
  - max_predicted_glucose
  - min_predicted_glucose
  - has_meal_taken_around_current_time
  - meal_recommendation
  - insulin_recommendation
  - exercise_recommendation
  - medication_recommendation
  - safety_notes

Your job is to evaluate the Output_Summary and ensure that they are:

1. Clinically reasonable
2. Within glucose safety limits
3. Logically consistent
4. Not harmful

You MUST:
- DO NOT recommend carbohydrates in the meal if the current blood glucose is higher than the desired range. In this case, only recommend protein and vegetables. 
- Remember that carobohydrate incrseses the blood glucose. If the meal is not taken and blood glucose if  >150, ensure right insulin dosage is taken before the delayed meal.
- Reject unsafe recommendations
- Explain WHY they are unsafe
- Suggest a safer alternative if possible

---

Safety Rules:

HYPOGLYCEMIA (current or predicted glucose <70 mg/dL):
- MUST recommend 15g fast-acting carbs
- MUST recommend recheck in 15 minutes
- MUST NOT recommend exercise
- MUST NOT recommend insulin

HYPERGLYCEMIA (current or predicted glucose  >180 mg/dL):
- Encourage hydration (500ml–1L water). It's will serve as a recommendation only. 
- Allow moderate exercise ONLY if <250 mg/dL
- If >250 mg/dL → avoid intense exercise

If both HYPOGLYCEMIA and HYPERGLYCEMIA happen in the predicted readings, ensure recommendations are aligned with avoiding what is happening first.

If meal is already taken within the last 1 hour, do not recommend another meal

INSULIN:
- Insulin is not recommended after the meal
- Must match glucose ranges at the time preferred meal:
  151–200 → 2 units
  201–250 → 4 units
  251–300 → 6 units
  301–350 → 8 units
  351–400 → 10 units
  >400 → advise contacting doctor

LOGICAL CONSISTENCY:
- Cannot recommend insulin + hypoglycemia treatment
- Cannot recommend exercise during hypoglycemia
- Meal + insulin timing must align

---

Output format (STRICT JSON):

{
"evaluated_output": {...},
  "safe": true/false,
  "violations": ["..."]

  
"""
)
print("✅ SafetyAgent created.")

✅ SafetyAgent created.


In [10]:
## Creating a dummy prediction model
# input: <min_past and max_past that automatically create random 24 past CGM data points within min & max range
# output: <min_pred, max_pred and randomly generated 12 future CGM data points within min and max range

import random

class DummyCGMModel:

    def predict(self, min_past, max_past):

        # Generate 24 past CGM values
        past_cgm = [random.randint(min_past, max_past) for _ in range(24)]

        # Define prediction range
        min_pred = min(past_cgm) - random.randint(5, 15)
        max_pred = max(past_cgm) + random.randint(5, 15)

        # Generate 12 future predictions
        future_cgm = [random.randint(min_pred, max_pred) for _ in range(12)]
        min_pred = min(future_cgm)
        max_pred = max(future_cgm)

        return {
            "past_cgm_24_points": past_cgm,
            "min_pred": min_pred,
            "max_pred": max_pred,
            "future_cgm_12_points": future_cgm
        }

# Save the model with joblib
import joblib

model = DummyCGMModel()

joblib.dump(model, "dummy_cgm_model.joblib")

print(" ✅ Model saved!")



 ✅ Model saved!


In [11]:
# prediction tool for the Main agent
import joblib
def predict_glucose(min_past,max_past):

    model = joblib.load("dummy_cgm_model.joblib")
    prediction = model.predict(min_past, max_past) # min max of past 24 values

    return prediction

In [47]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
#MainAgent_logger.info("Initializing Main agent")
Main_agent = Agent(
    name="Orchestrator_Agent",
    model=Gemini(
        model="gemini-2.5-flash",
        output_key="main_output",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""
System Role

You are a Blood Glucose Coaching Orchestrator Agent that helps users with diabetes maintain their blood glucose within the target range of 90–150 mg/dL.

Your job is to coordinate multiple specialized agents and tools to generate safe, personalized recommendations.

You do not guess medical information. Instead, you call the appropriate tools and synthesize their results into clear recommendations.

Safety Feedback Handling (VERY IMPORTANT)

If you receive input containing a field called `violations`:

- You MUST treat the previous Output_Summary as unsafe
- You MUST fix ALL listed violations by utilizing 'violation' info. 
- You MUST NOT repeat the same mistakes
- You MUST regenerate a fully safe recommendation

If violations exist, you should:
1. Identify what caused each violation using the 'violation' 
2. Adjust meal, insulin, exercise, and alerts accordingly
3. Ensure compliance with all safety rules before responding

Never ignore violations.

Primary Objective

Maintain the user’s glucose within 90–150 mg/dL by coordinating:

glucose prediction using glucose prediction tool

medication reminders

insulin dosing guidance

meal recommendations

exercise recommendations

hydration advice

Workflow (You MUST follow this order)

INPUT FORMAT:

You will receive input in one of the following formats:

1. Plain text (raw user input)

2. JSON object:
{
  "user_input": "...",
  "previous_output": {...},
  "violations": [...]
  
}
INSTRUCTIONS:

- If "user_input" exists → extract data from it
- If "evaluated_output" exists → use it as previous response
- If "violations" exist → fix issues before generating output

Always prioritize "user_input" for fresh data.

1. **Extract Key Information:** From the user's input, carefully extract the numerical values for `min_past` and `max_past` from the lines describing CGM readings. Also, extract the `current CGM reading`, `last time when the meal was taken`, `Current time`, `Weight`, `Height`, `Diet Preference`, `Usual meal time`, `Oral Medications dosing`, and `insulin intake`.
Predictions are for the next 1 hour from the current time at 5-minute intervals. has_meal_taken_around_current_time is deduced by closest preferred meal time, last meal taken, and the current time. For example, if the user doesnt indicate it has taken the lunch around his or her  preffered time, assume lunch is not taken and reccommend lunch. 

2. **Step 1 — Medication Alert:**
   - Check if the `Current time` corresponds to a `medication schedule` (e.g., 'pre-meal-before all 3 meals' and if a meal is upcoming or recently passed). If the `Oral Medications dosing` is 'pre-meal-before all 3 meals' and `Current time` is close to any `Usual meal time`.
   - If a medication is due, YOU MUST call the `AlertAgent` tool to notify the user about: oral medication, insulin, long acting insulin, GLP-1 agonist. Include specific medication type if applicable (e.g., 'oral medication before lunch').

3. **Step 2 — Predict Future Glucose:**
   - If already have past_cgm_24_points/min_pred/max_pred/
     future_cgm_12_points → skip this step entirely
   - Call predict_glucose EXACTLY ONCE
   - Store results and reuse across all remaining steps


4. **Step 3 — Insulin Dosage Recommendation:**
   
    - insulin` is 'yes' AND `has_meal_taken_around_current_time` is False:
         YOU MUST call the InsulinRecommenderAgent tool.
    - The tool returns a dose string like "Take 2 units of short acting insulin before meal"
   - YOU MUST parse the number from that string and populate:
     "insulin_recommendation": { "units": <extracted number>, "timing": "before <meal name>" }
   - NEVER set units to null if the tool returned a dose string.
   - If the tool returns {} (empty), set units to 0 and timing to "not required".
   - Provide `predicted glucose at the time of the meal`(from Step 2 prediction), OR `current CGM reading`if there is no indication that the meal is taken at the closest preferred time as input to the `InsulinAgent`tool.
   - InsulinAgent`tool provides a dosage recommendation based on the glucose level passed in the previou step


5. **Step 4 — Meal Recommendation:**
   - DO NOT recommend a meal if the user has already taken a meal within the last 1 hour, unless exception needs to made to avoid Hypoglycemia. You can still recommend hydration if applicable. 
   - Keep Diet preference in account
   - If the `Current time` is close to the user’s `Usual meal time` (breakfast, lunch, or dinner) OR if the `current CGM reading` indicates a need for immediate meal intervention (e.g., low glucose):
   - YOU MUST call the `MealAgent` tool. Provide the `predicted glucose trajectory` (from Step 2), `Diet Preference`, `current CGM reading`, and `glucose target range` (90–150 mg/dL) as input.
   - DO NOT recommend carbohydrates if the current blood glucose is higher than the desired range and has_meal_taken_around_current_time = False. In this case, only recommend protein and vegetables, as consuming carbohydrates will further increase blood glucose. 
   - The meal recommendation should help keep predicted glucose within 90–150 mg/dL.
   - Include hydration guidance if appropriate, based on the`current CGM reading`.



6. **Step 5 — Exercise Recommendation:**
   - YOU MUST call the `ExerciseAgent` tool to generate an exercise recommendation based on the required Calories burn to keep the glucose in the targetted range. 
   - Use: `predicted glucose trend` (from Step 2), `time since last meal`, and `safety constraints` (e.g., avoid exercise if glucose < 70 mg/dL).
   - Exercise recommendations should help maintain glucose within the target range.

Safety Rules (Always Follow)

Avoid recommending exercise if glucose is < 70 mg/dL.

Hypoglycemia protocol

If glucose < 70 mg/dL:

Eat fast-acting carbohydrates first

Wait 15 minutes

Recheck glucose

Repeat if still low

Medication timing

Pre-meal medication should be taken 15 minutes before the meal.

Exercise timing

Post-meal exercise should typically occur ~2 hours after eating.

Tool Usage Rules

You must follow these rules:

Do not fabricate glucose predictions.

Do not fabricate insulin dosage.

Do not invent nutritional values.

- If AlertAgent, MealAgent, or ExerciseAgent returns {} → treat as success, proceed.
- If InsulinRecommenderAgent returns {} → this means no insulin is required (glucose 
  in range). Set insulin_recommendation to { "units": 0, "timing": "not required" }.
- Never set insulin_recommendation to { "units": null, "timing": null }.
  null means unknown. 0 means not required. These are different.
- Never retry any tool already called in this workflow.


Final Response Format

After completing all tool calls, present a clear summary to the user containing:

1. Glucose Outlook

Brief description of past, current, and predicted glucose trend. Show relevant numbers in a chart. 
Include user inputs - `last time when the meal was taken`, `Current time`, `Weight`, `Height`, `Diet Preference`, `Usual meal time`, `Oral Medications dosing`, `insulin intake`, 'long acting Insulin intake'.

2. Medication Reminder

If applicable, show medication and Insulin dosage recommendations

3. Meal Recommendation

Suggested meal and hydration.

4. Exercise Recommendation

Activity type and timing.

5. Safety Notes

Any warnings about hypoglycemia or high glucose.

Keep explanations simple, supportive, and actionable.


CRITICAL OUTPUT RULES:

- Return ONLY valid JSON
- DO NOT use markdown (no ```json)
- DO NOT wrap JSON inside strings
- DO NOT nest JSON inside Output_Summary

Output MUST be exactly this JSON structure (no markdown, no extra keys):

{
  "Output_Summary": {
    "user_information": {
      "weight": "...",
      "height": "...",
      "diet": "...",
      "usual_meal_times": {...},
      "oral_medication": "...",
      "insulin": "...",
      "long acting Insulin":"...",
      "glp1": "..."
    },
    "current_glucose": <number>,
    "max_predicted_glucose": <number>,
    "min_predicted_glucose": <number>,
    "last_meal": "...",
    "current_time": "...",
    "has_meal_taken_around_current_time": true/false,
    "glucose_outlook": "...",
    "medication_recommendation": "...",
    "meal_recommendation": "...",
    "insulin_recommendation": { "units": <number>, "timing": "..." },
    "exercise_recommendation": "...",
    "safety_notes": "..."
  }
}


    """,
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[AgentTool(AlertAgent), FunctionTool(predict_glucose), AgentTool(InsulinAgent), AgentTool(MealAgent), AgentTool(ExerciseAgent)],
    # For the MainAgent, ensure that sub_agents are correctly defined before use
    #sub_agents=[AlertAgent, InsulinAgent, MealAgent, ExerciseAgent],
   
)
#MainAgent_logger.info(f"MainAgent Output: {MainAgent_output}")


print("✅ Main_agent created.")

✅ Main_agent created.


In [13]:
##Adding a Formatter Agent to generate a readable output
FormatterAgent = Agent(
    name="FormatterAgent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    output_key="formatted_output",
    instruction="""
You receive a validated JSON glucose management summary under the key "validated_output". 
-Make sure to also INCLUDE the quantity of Recommended Protein, Vegetables, and Carbohydrates.
-Be specific for what meal you are recommending: Breakfast, Lunch, or Dinner
-Convert it into a clean, friendly, easy-to-read report for a diabetes patient.

Format it EXACTLY as:

📊 Glucose Outlook
  Current Glucose: [current_glucose] mg/dL
  Predicted Range: [min_predicted_glucose] – [max_predicted_glucose] mg/dL
  Last Meal: [last_meal]
  Current Time: [current_time]

👤 User Information
  Weight: [weight] | Height: [height]
  Diet: [diet]
  Usual Meal Times: Breakfast [time] | Lunch [time] | Dinner [time]
  Oral Medication: [oral_medication] | Insulin: [insulin] | GLP-1: [glp1]

💊 Medication Reminder
  [medication_recommendation or "None due at this time"]

💉 Insuling Dosage recommendation
   [insulin_recommendation]

🍽️ Meal Recommendation

  [meal_recommendation]

🏃 Exercise Recommendation
  [exercise_recommendation]

⚠️ Safety Notes
  [safety_notes or "No concerns at this time"]

Rules:
- Output plain text ONLY
- No JSON, no markdown code blocks, no extra commentary
- Use simple, supportive, encouraging language
- No medical jargon
"""
)


print("✅ Formatter Agent created.")


✅ Formatter Agent created.


In [37]:
## New Controller: A2A synchronization of Main Agent and SafetyAgent (LLM-as-a-Judge) 
## feedback loop with MAX_RETRIES (run time input)

import json
import re
import csv
import os
import logging
from datetime import datetime
from google.adk.plugins.logging_plugin import LoggingPlugin
from google.adk.plugins import BasePlugin

# ─── TOKEN COUNTER PLUGIN ─────────────────────────────────────────────────────
# Fires on EVERY LLM call across all agents and sub-agents at any nesting depth

class TokenCounterPlugin(BasePlugin):

    def __init__(self):
        super().__init__(name="token_counter")
        self.input_tokens  = 0
        self.output_tokens = 0

    def reset(self):
        self.input_tokens  = 0
        self.output_tokens = 0

    async def after_model_callback(
        self, *, callback_context, llm_response
    ) -> None:
        meta = getattr(llm_response, "usage_metadata", None)
        if meta:
            self.input_tokens  += getattr(meta, "prompt_token_count",     0) or 0
            self.output_tokens += getattr(meta, "candidates_token_count", 0) or 0
        return None

# ─── SHARED PLUGIN INSTANCES ──────────────────────────────────────────────────
# One token_counter shared across all runners so all LLM calls are captured

token_counter = TokenCounterPlugin()


# ─── CSV LOGGING SETUP ────────────────────────────────────────────────────────

CSV_LOG_FILE = "agent_runs2.csv"
CSV_HEADERS  = [
    "timestamp",
    "duration_seconds",
    "input_tokens",
    "output_tokens",
    "is_safe",
    "attempts",
    "main_agent_output"
]

def init_csv_log():
    """Create CSV with headers if it doesn't exist yet."""
    if not os.path.exists(CSV_LOG_FILE):
        with open(CSV_LOG_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=CSV_HEADERS)
            writer.writeheader()

def append_csv_log(timestamp, duration_seconds, input_tokens, output_tokens, is_safe, attempts, main_agent_output):
    """Append one run's metrics to the CSV log."""
    with open(CSV_LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_HEADERS)
        writer.writerow({
            "timestamp":         timestamp,
            "duration_seconds":  round(duration_seconds, 3),
            "input_tokens":      input_tokens,
            "output_tokens":     output_tokens,
            "is_safe":           is_safe,
            "attempts":          attempts,
            "main_agent_output": json.dumps(main_agent_output) 
                                 if isinstance(main_agent_output, dict) 
                                 else str(main_agent_output)
        })

init_csv_log()  # ensure CSV + headers exist on import

# ─── HELPERS ──────────────────────────────────────────────────────────────────

def extract_text_from_debug(debug_result):
    try:
        if isinstance(debug_result, list):
            for event in reversed(debug_result):
                if hasattr(event, "content") and event.content:
                    return event.content.parts[0].text
        return str(debug_result)
    except Exception:
        return str(debug_result)

def extract_clean_summary(main_json):
    summary = main_json.get("Output_Summary", "")
    if not summary:
        return {}
    if isinstance(summary, dict):
        return summary
    summary = re.sub(r"```json|```", "", summary).strip()
    if summary.startswith("{"):
        try:
            inner = json.loads(summary)
            return inner.get("Output_Summary", inner)
        except Exception:
            pass
    return summary

# ─── MAIN CONTROLLER ──────────────────────────────────────────────────────────

async def run_main_with_safety(user_input):

    #Runners
    
    SafetyAgentrunner    = InMemoryRunner(agent=SafetyAgent,    plugins=[LoggingPlugin(), token_counter])
    FormatterAgentRunner = InMemoryRunner(agent=FormatterAgent, plugins=[LoggingPlugin(), token_counter])

    token_counter.reset()

    # ── Reset token counter for this run ─────────────────────────────────────
    token_counter.reset()

    # ── Capture run start time ────────────────────────────────────────────────
    run_timestamp = datetime.utcnow().isoformat(timespec="seconds") + "Z"
    run_start     = datetime.utcnow()

    # ── Loop state ────────────────────────────────────────────────────────────
    attempt                  = 0
    violations               = []
    corrected_recommendation = None
    clean_summary            = None
    final_is_safe            = False
    final_output             = None
    result                   = None

    while attempt < MAX_RETRIES:
        print(f"\n========== ATTEMPT {attempt + 1} ==========")

        # Main Agent runner
        MainAgentrunner      = InMemoryRunner(agent=Main_agent,     plugins=[LoggingPlugin(), token_counter])

        # --- BUILD MAIN AGENT PAYLOAD ---
        main_payload = {
            "user_input":      user_input,
            "previous_output": clean_summary if attempt > 0 else None,
            "violations":      violations    if attempt > 0 else []
        }

        # --- RUN MAIN AGENT ---
        # token_counter.on_llm_response fires for Main_agent + all its sub-agents
        raw_response = await MainAgentrunner.run_debug(json.dumps(main_payload))
        main_text    = extract_text_from_debug(raw_response)

        try:
            main_json = json.loads(main_text)
        except Exception:
            main_json = {"Output_Summary": main_text}

        print("\n--- MAIN OUTPUT ---")
        print(json.dumps(main_json, indent=2))

        # --- EXTRACT CLEAN SUMMARY ---
        clean_summary = extract_clean_summary(main_json)
        final_output  = clean_summary   # track latest main output for CSV

        print("\n--- CLEAN SUMMARY SENT TO SAFETY ---")
        print(json.dumps(clean_summary, indent=2) if isinstance(clean_summary, dict) else clean_summary)

        # --- RUN SAFETY AGENT ---
        # token_counter.on_llm_response fires for SafetyAgent calls
        safety_payload = json.dumps({"Output_Summary": clean_summary})
        raw_safety     = await SafetyAgentrunner.run_debug(safety_payload)
        safety_text    = extract_text_from_debug(raw_safety)

        print("\n--- RAW SAFETY TEXT ---")
        print(safety_text)

        try:
            safety_text_clean = re.sub(r"```json|```", "", safety_text).strip()
            safety_result     = json.loads(safety_text_clean)
        except Exception:
            safety_result = {
                "safe":                     False,
                "violations":               ["Could not parse safety agent response"],
                "corrected_recommendation": None
            }

        print("\n--- SAFETY OUTPUT ---")
        print(json.dumps(safety_result, indent=2))

        safe                     = safety_result.get("safe", False)
        new_violations           = safety_result.get("violations", [])
        corrected_recommendation = safety_result.get("corrected_recommendation", None)

        # --- CHECK IF VIOLATIONS ARE REPEATING (agent is stuck) ---
        if not safe and new_violations == violations and attempt > 0:
            final_is_safe = False
            result = {
                "status":      "failed",
                "reason":      "Repeated violations — agent is stuck",
                "violations":  new_violations,
                "last_output": clean_summary
            }
            break

        violations = new_violations

        # --- SAFE: RUN FORMATTER AND RETURN ---
        if safe:
            # token_counter.on_llm_response fires for FormatterAgent calls
            fmt_payload = json.dumps({"validated_output": clean_summary})
            raw_fmt     = await FormatterAgentRunner.run_debug(fmt_payload)
            readable    = extract_text_from_debug(raw_fmt)

            final_is_safe = True
            result        = readable
            attempt += 1
            break

        attempt += 1

    # --- MAX RETRIES EXHAUSTED ---
    if result is None:
        final_is_safe = False
        result = {
            "status":      "failed",
            "reason":      "Max retries exceeded",
            "violations":  violations,
            "last_output": clean_summary
        }

    # ── Write metrics to CSV ──────────────────────────────────────────────────
    # token_counter now holds the TRUE total across all agents, sub-agents,
    # and every retry attempt in this run
    duration = (datetime.utcnow() - run_start).total_seconds()
    append_csv_log(
        timestamp         = run_timestamp,
        duration_seconds  = duration,
        input_tokens      = token_counter.input_tokens,
        output_tokens     = token_counter.output_tokens,
        is_safe           = final_is_safe,
        attempts          = attempt ,
        main_agent_output = final_output
    )

    return result

In [15]:
#### User Input ####

user_input = """
min_past = 90
max_past = 200
current_glucose = 165

last_meal = Breakfast at 7:00 AM ET
current_time = 1:30 PM ET

weight = 75 kg
height = 1.65 m
diet = Non-Veg

usual_meal_times:
  breakfast = 7:00 AM ET
  lunch = 12:30 PM ET
  dinner = 7:00 PM ET

oral_medication = pre-meal
insulin = yes
long_acting_insulin = Yes, every night 9PM ET
glp1 = no
"""

In [49]:
# Run Command with MAX_RETRIES 

MAX_RETRIES = 2
import time

start_time = time.time()
result = await run_main_with_safety(user_input)

end_time = time.time()
elapsed_time = end_time - start_time
print(result)
print(f"✅ Main_agent took {elapsed_time:.2f} seconds to generate the response.")




========== ATTEMPT 1 ==========

 ### Created new session: debug_session_id

User > {"user_input": "\nmin_past = 90\nmax_past = 200\ncurrent_glucose = 165\n\nlast_meal = Breakfast at 7:00 AM ET\ncurrent_time = 1:30 PM ET\n\nweight = 75 kg\nheight = 1.65 m\ndiet = Non-Veg\n\nusual_meal_times:\n  breakfast = 7:00 AM ET\n  lunch = 12:30 PM ET\n  dinner = 7:00 PM ET\n\noral_medication = pre-meal\ninsulin = yes\nlong_acting_insulin = Yes, every night 9PM ET\nglp1 = no\n", "previous_output": null, "violations": []}
[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-8a6fd7aa-8ef1-4477-8c6f-b5847935615e
[logging_plugin]    Session ID: debug_session_id
[logging_plugin]    User ID: debug_user_id
[logging_plugin]    App Name: InMemoryRunner
[logging_plugin]    Root Agent: Orchestrator_Agent
[logging_plugin]    User Content: text: '{"user_input": "\nmin_past = 90\nmax_past = 200\ncurrent_glucose = 165\n\nlast_meal = Breakfast at 7:00 AM ET\ncurrent_time = 1:30 PM ET\n\n

In [ ]:
# Current bugs
"""
1. Doesnt read right min and max of predicted readings --fixed
2. Current time can be fetched systamtically in the user_input --> Update AlertAgent and user_inputs accordingly
3. Too many tokens are being passed around Pass what is absolutely needed as new inputs
"""
